# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy scipy soundfile matplotlib --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import numpy as np, soundfile as sf
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
from IPython.display import Audio, display

**Declarations**

Set the variables below and run the generation cell. 

- `FILE_A` is the path to the first audio file that will be analysed, e.g. `./audio1.wav`.
- `FILE_B` is the path to the first audio file that will be analysed, e.g. `./audio2.wav`.

In [ ]:
# --- Declarations: editable variables for students ---
FILE_A = ...
FILE_B = ...

### 1. Theory
#### 1.1 Setup
This Notebook allows you to compare two audio files (FILE_A and FILE_B) in time and frequency domain, as well as a spectogram comparison.

In [ ]:
# Helper Functions + Prints
db = lambda a: 20*np.log10(np.maximum(a, 1e-12))

xo, fs = sf.read(FILE_A, always_2d=True); xm, fs2 = sf.read(FILE_B, always_2d=True)
assert fs==fs2, 'Sample rate mismatch'
L = min(int(5*fs), len(xo), len(xm))
xo, xm = xo[:L], xm[:L]

for arr in (xo, xm):
    peak = np.max(np.abs(arr), axis=0, keepdims=True); peak[peak==0]=1; arr /= peak
t = np.arange(L)/fs

print(FILE_A)
display(Audio(filename=FILE_A))
print(FILE_B)
display(Audio(filename=FILE_B))


#### 1.2 Time Domain
Shows how the amplitude of the sound changes over time. This is what we perceive as loudness and rhythm.

In [ ]:
# Time Domain Signal Plot
fig, ax = plt.subplots(1,2,figsize=(10,3),sharey=True,sharex=True)
for a,Y,n in ((ax[0],xo, FILE_A),(ax[1],xm,FILE_B)):
    a.plot(t, Y[:,0], lw=0.9, color="#202A63")
    if Y.shape[1]>1: a.plot(t, Y[:,1], lw=.9)
    a.set_title(n); a.set_xlabel('Time [s]')
ax[0].set_ylabel('Amplitude')
plt.tight_layout(); plt.show()

#### 1.3 Frequency Domain
Displays how much energy the signal has at each frequency, helping to identify dominant tones or bands.

In [ ]:
# Frequency Domain Signal Plot
def quick_fft(x):
    N=len(x); w=np.hanning(N); X=np.fft.rfft(x*w); f=np.fft.rfftfreq(N,1/fs)
    mag=np.abs(X); mag[0]/=2; mag=mag/(N/2)/(w.sum()/N); return f, db(mag)

fig, ax = plt.subplots(1,2,figsize=(10,3),sharey=True,sharex=True)
for a,Y,n in ((ax[0],xo,FILE_A),(ax[1],xm,FILE_B)):
    for ch in range(Y.shape[1]):
        f,m = quick_fft(Y[:,ch]); a.plot(f/1000, m, lw=.9, color='#8E0E05')
    a.set_title(n); a.set_xlabel('Freq [kHz]')
ax[0].set_ylabel('Mag [dBV]')
ax[0].set_xlim(0, (fs/2)/1000)
plt.tight_layout(); plt.show()

#### 1.4 Spectogram
Shows how the frequency content of the sound changes over time, revealing patterns such as beats or encoded data.

In [ ]:
# Spectogram Plot
def spec(x): f,t,S = spectrogram(x, fs=fs, nperseg=256, noverlap=128, window='hann'); return f,t,db(S)
f1,t1,S1 = spec(xo[:,0]); f2,t2,S2 = spec(xm[:,0])
fig,(a0,a1)=plt.subplots(1,2,figsize=(12,3.5),sharey=True)
im0=a0.pcolormesh(t1,f1,S1,cmap='viridis',shading='gouraud'); a0.set_title(FILE_A)
im1=a1.pcolormesh(t2,f2,S2,cmap='viridis',shading='gouraud'); a1.set_title(FILE_B)
for a in (a0,a1): a.set_xlabel('Time [s]')
a0.set_ylabel('Freq [Hz]')
fig.colorbar(im1, ax=[a0,a1], shrink=.85, label='Mag [dB]')
plt.show()